# 🩺 MedOrqis Medical Foundation Model — 100% Free Cloud Training
### Fine-Tuning `Applied-Innovation-Center/Karnak-6B-v1.0` via QLoRA (4-bit NF4)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/seotarek/MedOrqis/blob/main/training/MedOrqis_Free_Training_Colab.ipynb)

هذا الدفتر مخصص لتدريب نموذج **MedOrqis-Karnak-6B** مجاناً بالكامل 100% على كارت الشاشة المجاني **NVIDIA T4** في Google Colab.

- **التكلفة**: **0$ (مجاني تماماً)**
- **استهلاك جهازك/إنترنتك**: **0 بايت (كل التحميل والتدريب سحابي بالكامل داخل سيرفرات جوجل)**
- **الناتج**: رفع محولات LoRA المدربة مباشرة إلى حسابك على Hugging Face: `seotarek/medorqis-karnak-6b-lora`

## 1. فحص كارت الشاشة المجاني (GPU Verification)

In [ ]:
!nvidia-smi

## 2. تثبيت المكتبات السحابية المطلوبة (Dependencies)

In [ ]:
!pip install -q -U torch transformers peft trl bitsandbytes datasets accelerate huggingface_hub

## 3. تسجيل الدخول إلى Hugging Face

In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from huggingface_hub import login

# الحصول على توكن Hugging Face من Secrets أو الإدخال المباشر
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    pass

if not HF_TOKEN:
    HF_TOKEN = os.getenv('HF_TOKEN')

if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass('أدخل توكن Hugging Face الخاص بك (Write Token): ')

login(token=HF_TOKEN)

BASE_MODEL_ID = "Applied-Innovation-Center/Karnak-6B-v1.0"
DATASET_ID = "seotarek/medorqis-clinical-instructions"
OUTPUT_LORA_REPO = "seotarek/medorqis-karnak-6b-lora"

print(f"Target Base Model: {BASE_MODEL_ID}")
print(f"Target Output Hub: {OUTPUT_LORA_REPO}")
print(f"CUDA Device: {torch.cuda.get_device_name(0)} (VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB)")

## 4. تحميل النموذج الأساسي بضغط 4-bit NF4 (Quantized Base Model)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 5. تجهيز الداتاسيت الطبية من Hugging Face

In [ ]:
dataset = load_dataset(DATASET_ID, split="train", token=HF_TOKEN)

def format_example(ex):
    messages = ex.get("messages", [])
    tools = ex.get("tools", None)
    try:
        text = tokenizer.apply_chat_template(messages, tools=tools, tokenize=False, add_generation_prompt=False)
    except Exception:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

formatted_dataset = dataset.map(format_example)
print(f"Loaded {len(formatted_dataset)} clinical instruction samples.")

## 6. بدء التدريب السحابي (SFTTrainer Training)

In [ ]:
training_args = TrainingArguments(
    output_dir="./medorqis_karnak6b_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=2,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    dataset_text_field="text",
    max_seq_length=1536,
    tokenizer=tokenizer,
    args=training_args,
    peft_config=peft_config
)

print("Starting Free Cloud Training...")
trainer.train()

## 7. رفع الموديل النهائي مباشرة إلى Hugging Face Hub

In [ ]:
save_dir = "./medorqis-karnak-6b-lora-final"
trainer.model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print(f"Uploading LoRA Adapter to https://huggingface.co/{OUTPUT_LORA_REPO} ...")
trainer.model.push_to_hub(OUTPUT_LORA_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(OUTPUT_LORA_REPO, token=HF_TOKEN)
print("\n🎉 ألف مبروك! تم تدريب النموذج ورفعه لايف بنجاح 100% مجاناً!")